<a href="https://colab.research.google.com/github/vyron-arvanitis/pytorch_tutorials/blob/main/Copy_of_03_pytorch_copmputer_vision.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Computer vision libraries in PyTorch

* [`torchvision`](https://docs.pytorch.org/vision/stable/index.html) -  base library for PyTorch computer vision
* `torchvision.datasets` - datasets and data loading functions for computer vision
* `torchvision.models` - get pretrained computer vision models
* `torchvision.transforms` - functions manipulating vision data
* `torch.utils.data.Dataset` - base dataset class for PyTorch
* `torch.utils.data.Dataloader` - creates a python iterable over a dataset

In [ ]:
# Impoty PyTorch
import torch
from torch import nn

# Import torchvision
import torchvision
from torchvision import datasets
from torchvision import transforms
from torchvision.transforms import ToTensor

# Import matplotlib
import matplotlib.pyplot as plt

# Import numpy
import numpy as np

print(torch.__version__)
print(torchvision.__version__)

## Get a dataset

FashionMnist will be used, from [`torchvision.dataseset`](https://docs.pytorch.org/vision/main/datasets.html)

In [ ]:
# Setup training data
train_data = datasets.FashionMNIST(
    root="data", # where the data are downloaded to
    train=True,
    download=True, # donwload the data yes!
    transform=ToTensor(),
    target_transform=None
)

test_data = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=ToTensor(),
    target_transform=None
)

In [ ]:
len(train_data), len(test_data), type(train_data), type(train_data[0]), len(train_data[0]), type(train_data[0][0]), train_data[0][0].shape, type(train_data[0][1]), train_data[0][1]

In [ ]:
dir(train_data)

In [ ]:
type(vars(train_data))

In [ ]:
train_data.data

In [ ]:
len(train_data.data), train_data.data.shape

In [ ]:
(train_data.data[0]).dtype , train_data[0][0].dtype

In [ ]:
raw = train_data.data[0]
x, y = train_data[0]

print(raw.dtype, raw.shape)
print(x.dtype, x.shape)


In [ ]:
# See the train data
image, label = train_data[0]
image, label

In [ ]:
class_names = train_data.classes
class_names

In [ ]:
class_to_idx = train_data.class_to_idx
class_to_idx

In [ ]:
train_data.targets

In [ ]:
# Check the shape of the image
print(f"The shape of hte image is {image.shape}")
print(f"Iamge lable: {class_names[label]}")

# Visualize the iamges

In [ ]:
import matplotlib.pyplot as plt
iamge, label = train_data[0]

# plt.imshow(image) # Invalid shape (1, 28, 28) for image data
plt.imshow(image.squeeze(), cmap="grey")
plt.title(class_names[label])

In [ ]:
# Plot more images
torch.manual_seed(42)
fig = plt.figure(figsize=(9, 9))
rows, cols = 4, 4
for i in range(1, rows*cols+1):
  random_idx = torch.randint(0, len(train_data), size=[1]).item() #.item() to get the value of the tensor object, randint Returns a tensor filled with random integers
  img, label = train_data[random_idx]
  fig.add_subplot(rows, cols, i)
  plt.imshow(img.squeeze(), cmap="grey")
  plt.title(class_names[label])
  plt.axis("off");
  i +=1

In [ ]:
train_data

## Prepare Dataloader

Dataloaders turn the dataset into a Python iterable.
We wish to turn them into **batches** ( or min-batches)

In [ ]:
from torch.utils.data import DataLoader

BATCH_SIZE = 32
train_dataloader = DataLoader(dataset=train_data,
                              batch_size=BATCH_SIZE,
                              shuffle=True)

test_dataloader = DataLoader(dataset=test_data,
                             batch_size=BATCH_SIZE,
                             shuffle=False) # we do not shuffle the test data, as they are never seen by the model so no need!
train_dataloader, test_dataloader

In [ ]:
dir(train_dataloader)

In [ ]:
print(f" Length of train_dataloader : {len(train_dataloader)} batches of {BATCH_SIZE}")
print(f" Length of test_dataloader : {len(test_dataloader)} batches of {BATCH_SIZE}")

In [ ]:
print(type(train_dataloader.dataset))

In [ ]:
# What is inside?
train_features_batch, train_labels_batch = next(iter(train_dataloader))
train_features_batch.shape, train_labels_batch.shape

In [ ]:
# What is inside?
# train_features_batch, train_labels_batch = train_dataloader.dataset[0]
# train_features_batch.shape, train_labels_batch

In [ ]:
# Show a sample
torch.manual_seed(42)
random_idx = torch.randint(0, len(train_features_batch), size=[1]).item()
img, label = train_features_batch[random_idx], train_labels_batch[random_idx]
plt.imshow(img.squeeze(), cmap="grey")
plt.title(class_names[label])
plt.axis("off")

## First model, base model (baseline model)

In [ ]:
# Create a Flatten layer
flatten_model = nn.Flatten()

# Get a single sample
x = train_features_batch[0]

# Flatten the sample
output = flatten_model(x) # __call__ of flatter -> forward pass
print(f"The shape before flattening is {x.shape}") # C,H,W
print(f"The shape after flatterning is {output.shape}") # C,H*W

In [ ]:
class FashionMNISTModelV0(nn.Module):
    def __init__(self, hidden_units: int, output_shape: int):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Flatten(),
            nn.LazyLinear(hidden_units),   # infers in_features on first forward
            nn.Linear(hidden_units, output_shape),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.layers(x)

In [ ]:
model_0 = FashionMNISTModelV0(hidden_units=10,
                              output_shape=len(class_names) # one for every class
                              ).to("cpu")
model_0

In [ ]:
dummy_x = torch.rand([1, 1, 28, 28])
model_0(dummy_x).argmax(dim=1)

In [ ]:
model_0.state_dict()

### Setup loss optimizer and evaluation metrics

* Loss function -we have **multiclass** therefore we wil use the `nn.CrossEntropyLoss()`
* Optimizer - `torch.optim.SGD()` or mayube `torch.optim.Adam()`
* We will use `accuracy` as valuation metric

In [ ]:
# Rewrite again the accuracy_fn by yourself
def accuracy_fn(y_true: torch.Tensor, y_pred: torch.Tensor):
  truth_sum = torch.sum(y_true == y_pred).item()
  return truth_sum/(len(y_true))* 100

In [ ]:
y_true = torch.arange(1, 5, 1)
y_pred = torch.arange(1,5, 1)
acc = accuracy_fn(y_true, y_pred)
acc

In [ ]:
import requests
from pathlib import Path

if Path("helper_functions.py").is_file():
  print("helper_functions.py already exists, skipping download...")
else:
  req = requests.get("https://raw.githubusercontent.com/mrdbourke/pytorch-deep-learning/refs/heads/main/helper_functions.py")
  with open("helper_functions.py", "wb") as f:
    f.write(req.content)

In [ ]:
# Setup loss and optimizer
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model_0.parameters(),
                            lr=0.001)

### Create a function to time the experiment

We care about 2 things
1. Model performance
2. How fast it runs

In [ ]:
from timeit import default_timer as timer
def print_train_time(start: float,
                     end: float,
                     device: torch.device=None):
  """Prints the total Runtime of the model"""
  total_time = end - start
  print(f"Train time on {device}: {total_time:.3f} seconds")
  return total_time

In [ ]:
start_time = timer()
end_time = timer()
print_train_time(start_time, end_time, "cpu")

In [ ]:
len(train_dataloader), len(train_dataloader.dataset)

### Create a training loop and train a model on batches of data

1. Loop through epochs
2. loop through training batches, perform training steps, calualte the trian loss **per batch**
3. Loop through test batches, perform testing steps, calucalte the test loss **per batch**
4. Print out what's happening
5. Time it all

In [ ]:
# Import tqdm
from tqdm.auto import tqdm

# Set the seed and start the timer
torch.manual_seed(42)
torch.cuda.manual_seed(42)
train_time_start_on_cpu = timer()

# Se the number of epochs
epochs = 3

# create training loop
for epoch in tqdm(range(epochs)):
  print(f"Epoch: {epoch} \n-----")

  ### Training
  train_loss = 0
  # Loop through training batches
  for batch, (X, y) in enumerate(train_dataloader):
    model_0.train()

    y_pred = model_0(X)

    loss = loss_fn(y_pred, y)
    train_loss += loss

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if batch % 400 == 0:
      print(f"Looked at {batch * len(X)}/{len(train_dataloader.dataset)} samples")

  # divide total train loss by length of the train datalader
  train_loss /= len(train_dataloader) # len(train_dataloader) is the number of batches

  ### Testing
  test_loss, test_acc = 0,0
  model_0.eval()
  with torch.inference_mode():
    for X_test, y_test in test_dataloader:
      test_pred = model_0(X_test)

      test_loss += loss_fn(test_pred, y_test)
      test_acc += accuracy_fn(y_true=y_test,
                              y_pred=test_pred.argmax(dim=1))

    # Caluclate test loss average per batch
    test_loss /= len(test_dataloader)
    # Calualte the test acc average per batch
    test_acc /= len(test_dataloader)

  print(f"Train loss: {train_loss:.5f} | Test loss: {test_loss:.4f} | Test acc: {test_acc:.2f}%")

  # Calcualte the training time
  train_time_end_on_cpu = timer()
  total_train_time_model_0 = print_train_time(start=train_time_start_on_cpu, end=train_time_end_on_cpu, device=str(next(model_0.parameters()).device))





### Make predictions and get Model 0 results

In [ ]:
torch.manual_seed(42)
def eval_model(model: torch.nn.Module,
              data_loader: torch.utils.data.DataLoader,
              loss_fn: torch.nn.ModuleDict,
              accuracy_fn):
  """Returns a dictionary contatinig the reesults of a model predicting on data_loader
  """

  loss, acc = 0, 0
  model.eval()
  with torch.inference_mode():
    for X, y in tqdm(data_loader):
      y_pred = model(X)
      loss += loss_fn(y_pred, y)
      acc += accuracy_fn(y_true=y, y_pred=y_pred.argmax(dim=1))
    # Scale the loss and acc to fin the avg loss and acc per batch
    loss /= len(data_loader)
    acc /= len(data_loader)

  return {"model_name": model.__class__.__name__,
          "model_loss": loss.item(),
          "model_acc": acc
          }



In [ ]:
model_0_results = eval_model(model=model_0,
                             data_loader=test_dataloader,
                             loss_fn=loss_fn,
                             accuracy_fn=accuracy_fn)
model_0_results